# A1.13 · Repudiation and untraceability

**Function A — Security Architecture & Platform → The Agentic Reference Architecture, and Every Risk It Carries**  ·  *Security of AI*

Builds on **[A1.12 · Resource overload](https://spbreed.github.io/cyber-commons/lessons/A1.12.html)**.

| | |
|---|---|
| Open-source tooling | OpenTelemetry |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


**OWASP T8 — Repudiation & Untraceability.**

The **observability** component decides whether anything that just happened can
be explained. Most agent logging records tool calls: which tool, what arguments,
what came back. That is enough to debug the agent and not enough to investigate
it.

Three fields are usually missing, and each one removes a different question from
the set you can answer.

**The human principal.** Without it, "which user caused this?" has no answer —
the log says `agent-svc`, as in A1.6.

**The motivating input.** The tool call is recorded; the thing that made the
agent decide to call it is not. So root cause cannot be established at all. You
can see that the agent emailed a file, and nothing tells you the retrieved
document that told it to.

**The delegation chain.** In a multi-agent topology, which hop originated this?
Without the chain you have a set of actions and no order.

There is a fourth problem that is structural rather than about fields: **if the
agent can write to the log store, the log is not evidence.** An agent with
credentials broad enough to be interesting usually has credentials broad enough
to touch the observability stack, and nobody notices until they need the record
to be trustworthy.

> **Where this lands on the reference architecture.**
>
> ```
> ingress -> orchestrator -> agent_runtime -> model
>                                |              |
>                          messaging        tools / mcp
>                                |              |
>                       knowledge / memory   egress
>            identity + policy wrap every call · observability records it
> ```
\n## 2 · The risk, realised\n\nA deletion happened. Answer three questions from the log you have.

In [ ]:
LOG = [
 {"ts": "09:14:02", "actor": "agent-svc", "tool": "search",     "args": {"q": "invoice 8812"}},
 {"ts": "09:14:07", "actor": "agent-svc", "tool": "fetch_doc",  "args": {"id": "wiki/473"}},
 {"ts": "09:14:11", "actor": "agent-svc", "tool": "run_query",  "args": {"sql": "DELETE FROM invoices WHERE id=8812"}},
 {"ts": "09:14:12", "actor": "agent-svc", "tool": "send_email", "args": {"to": "ops@corp.example"}},
]

print("the log you have:")
for e in LOG:
    print(f"   {e['ts']}  {e['actor']:10s}{e['tool']:12s}{e['args']}")

QUESTIONS = {
 "which user caused the deletion?":            "principal",
 "what made the agent decide to delete?":      "motivating_input",
 "which agent in the chain originated it?":    "delegation_chain",
}
print()
print(f"{'question':44s}{'field needed':20s}present?")
answerable = 0
for q, field in QUESTIONS.items():
    present = any(field in e for e in LOG)
    answerable += present
    print(f"{q:44s}{field:20s}{'yes' if present else 'NO'}")

print(f"\nquestions answerable from this log: {answerable}/{len(QUESTIONS)}")
print()
print("The log is not broken. It is complete for debugging and empty for")
print("investigation, and the difference is three fields nobody was asked for.")
print()
print("One more: agent-svc holds db:admin. The log store is a database.")
print("A record the actor can edit is not evidence of anything.")
assert answerable == 0

## What you just proved

A complete-looking tool-call log answers none of the three questions an investigation needs — which user, what motivated it, which hop originated it — because the principal, the motivating input and the delegation chain were never recorded.

## Your turn

Take yesterday's agent logs and try to answer 'which user caused this action'. Time how long it takes. That number is your time-to-attribution during an incident, when it will be worse.

---

**Next → [A1.14 · Overwhelming the human in the loop](https://spbreed.github.io/cyber-commons/lessons/A1.14.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.13.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.13.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*